# Industrial Helmet Detection - Latest YOLO11 Model Training Pipeline

Self-contained pipeline engineered for high-speed GPU execution on Google Colab or local GPU.
- Architecture: **YOLO11 Nano (`yolo11n.pt`)**
- Dataset: 5,000 real-world annotated safety helmet images from Kaggle (`andrewmvd/hard-hat-detection`)
- Target Classes: `helmet` (Safe, Class 0), `no_helmet` (Violation, Class 1), `person` (Context, Class 2)

### Step 1: Install Required Libraries & Verify GPU

In [ ]:
!pip install -q ultralytics kagglehub opencv-python pillow pyyaml

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Acceleration Active: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

### Step 2: Download the 5,000 Helmet Images Directly from Kaggle

In [ ]:
import kagglehub

print("Downloading 5,000 helmet images from Kaggle...")
raw_dataset_dir = kagglehub.dataset_download('andrewmvd/hard-hat-detection')
print(f"Raw dataset location: {raw_dataset_dir}")

### Step 3: Convert Annotations & Build Train/Val Splits

In [ ]:
import glob
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

# Setup working data directories
BASE_DIR = Path('dataset_prepared')
TRAIN_IMG = BASE_DIR / 'train' / 'images'
TRAIN_LBL = BASE_DIR / 'train' / 'labels'
VAL_IMG = BASE_DIR / 'val' / 'images'
VAL_LBL = BASE_DIR / 'val' / 'labels'

for p in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    p.mkdir(parents=True, exist_ok=True)

# Class dictionary
CLASS_MAP = {'helmet': 0, 'head': 1, 'person': 2}

def convert_box(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

src_annotations = Path(raw_dataset_dir) / 'annotations'
src_images = Path(raw_dataset_dir) / 'images'

xml_files = sorted(glob.glob(str(src_annotations / '*.xml')))
print(f"Total annotation files found: {len(xml_files)}")

# Split 80% train, 20% validation
split_idx = int(len(xml_files) * 0.8)
train_files = xml_files[:split_idx]
val_files = xml_files[split_idx:]

def process_split(file_list, img_dest, lbl_dest):
    count = 0
    for xml_path in file_list:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        filename = root.find('filename').text
        if not filename.endswith(('.png', '.jpg', '.jpeg')):
            filename = Path(xml_path).stem + '.png'
            
        src_img = src_images / filename
        if not src_img.exists():
            continue
            
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)
        if w == 0 or h == 0:
            continue
            
        yolo_lines = []
        for obj in root.iter('object'):
            cls_name = obj.find('name').text.lower().strip()
            if cls_name not in CLASS_MAP:
                continue
            cls_id = CLASS_MAP[cls_name]
            bndbox = obj.find('bndbox')
            xmin = float(bndbox.find('xmin').text)
            xmax = float(bndbox.find('xmax').text)
            ymin = float(bndbox.find('ymin').text)
            ymax = float(bndbox.find('ymax').text)
            bbox = convert_box((w, h), (xmin, xmax, ymin, ymax))
            yolo_lines.append(f"{cls_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}")
            
        if yolo_lines:
            shutil.copy2(src_img, img_dest / filename)
            lbl_name = Path(filename).stem + '.txt'
            with open(lbl_dest / lbl_name, 'w') as f:
                f.write('\n'.join(yolo_lines) + '\n')
            count += 1
    return count

print("Converting train split (4,000 images)...")
n_train = process_split(train_files, TRAIN_IMG, TRAIN_LBL)
print(f"Prepared {n_train} training samples.")

print("Converting validation split (1,000 images)...")
n_val = process_split(val_files, VAL_IMG, VAL_LBL)
print(f"Prepared {n_val} validation samples.")

### Step 4: Generate `data.yaml` Specification

In [ ]:
import yaml
import os

yaml_config = {
    'path': str(BASE_DIR.resolve()),
    'train': 'train/images',
    'val': 'val/images',
    'names': {
        0: 'helmet',
        1: 'no_helmet',
        2: 'person'
    }
}

yaml_path = 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_config, f, default_flow_style=False)

print(f"data.yaml created successfully at: {os.path.abspath(yaml_path)}")

### Step 5: Train the Latest YOLO11 Model for 50 Epochs

In [ ]:
from ultralytics import YOLO

# Load the latest pre-trained YOLO11 architecture
model = YOLO('yolo11n.pt')

# Train for 50 epochs on the 5,000 helmet dataset
results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='helmet_yolo11_run',
    project='runs_output'
)

print("YOLO11 Model Training Complete!")

### Step 6: Export Trained Weights

### Step 6: Visual Performance Metrics & Evaluation

In [ ]:
# =========================================================
# Industry-Standard Performance Evaluation & Metrics Report
# =========================================================
import os
import matplotlib.pyplot as plt
from PIL import Image

def find_file(filename):
    candidates = [
        filename,
        os.path.join('runs', 'detect', 'val', filename),
        os.path.join('..', 'runs', 'detect', 'val', filename),
        os.path.join(r'C:\Users\rithv\Downloads\helmet detection industry track\notebooks', filename),
        os.path.join(r'C:\Users\rithv\Downloads\helmet detection industry track\runs\detect\val', filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

r_path = find_file('BoxR_curve.png')
cm_path = find_file('confusion_matrix_normalized.png') or find_file('confusion_matrix.png')

# Industry-Standard Evaluation Report
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

if r_path:
    axes[0].imshow(Image.open(r_path))
    axes[0].axis('off')
    axes[0].set_title('Recall-Confidence Curve', fontsize=14, fontweight='bold')

if cm_path:
    axes[1].imshow(Image.open(cm_path))
    axes[1].axis('off')
    axes[1].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
